In [1]:
import pandas as pd
import numpy as np
import joblib
import gradio as gr

D:\ML\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = "../models/final_elastic_net_model.pkl"

model = joblib.load(model_path)

print("Final model loaded successfully")

Final model loaded successfully


In [4]:
data_path = "../../Dataset/processed/cleaned_movie_data.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (1696, 14)


In [5]:
release_periods = sorted(
    df["Release_Period"].dropna().unique().tolist()
)

genres = sorted(
    df["Genre"].dropna().unique().tolist()
)

lead_stars = sorted(
    df["Lead_Star"].dropna().unique().tolist()
)

directors = sorted(
    df["Director"].dropna().unique().tolist()
)

music_directors = sorted(
    df["Music_Director"].dropna().unique().tolist()
)

print("Release periods:", release_periods)
print("Genres:", genres)
print("Number of lead stars:", len(lead_stars))
print("Number of directors:", len(directors))
print("Number of music directors:", len(music_directors))

Release periods: ['Holiday', 'Normal']
Genres: ['action', 'adult', 'animation', 'comedy', 'documentary', 'drama', 'fantasy', 'horror', 'love_story', 'masala', 'mythological', 'rom__com', 'suspense', 'thriller']
Number of lead stars: 764
Number of directors: 1048
Number of music directors: 630


In [6]:
categorical_columns = [
    "Release_Period",
    "Whether_Remake",
    "Whether_Franchise",
    "Genre",
    "New_Actor",
    "New_Director",
    "New_Music_Director",
    "Lead_Star",
    "Director",
    "Music_Director"
]

numerical_columns = [
    "Number_of_Screens",
    "Budget(INR)"
]

In [7]:
training_features = df.drop(
    columns=["Movie_Name", "Revenue(INR)"]
)

training_encoded = pd.get_dummies(
    training_features,
    columns=categorical_columns,
    drop_first=True
)

training_encoded = training_encoded.astype(float)

feature_columns = training_encoded.columns.tolist()

print("Number of model features:", len(feature_columns))

Number of model features: 2460


In [8]:
def predict_revenue(
    release_period,
    whether_remake,
    whether_franchise,
    genre,
    new_actor,
    new_director,
    new_music_director,
    lead_star,
    director,
    music_director,
    number_of_screens,
    budget
):
    movie_data = pd.DataFrame([{
        "Release_Period": release_period,
        "Whether_Remake": whether_remake,
        "Whether_Franchise": whether_franchise,
        "Genre": genre,
        "New_Actor": new_actor,
        "New_Director": new_director,
        "New_Music_Director": new_music_director,
        "Lead_Star": lead_star,
        "Director": director,
        "Music_Director": music_director,
        "Number_of_Screens": number_of_screens,
        "Budget(INR)": budget
    }])

    movie_encoded = pd.get_dummies(
        movie_data,
        columns=categorical_columns,
        drop_first=True
    )

    movie_encoded = movie_encoded.reindex(
        columns=feature_columns,
        fill_value=0
    )

    movie_encoded = movie_encoded.astype(float)

    prediction = model.predict(movie_encoded)[0]

    if prediction < 0:
        prediction = 0

    return f"₹{prediction:,.2f}"

In [9]:
test_prediction = predict_revenue(
    release_period=release_periods[0],
    whether_remake="No",
    whether_franchise="No",
    genre=genres[0],
    new_actor="No",
    new_director="No",
    new_music_director="No",
    lead_star=lead_stars[0],
    director=directors[0],
    music_director=music_directors[0],
    number_of_screens=5,
    budget=85000
)

print("Predicted Revenue:", test_prediction)

Predicted Revenue: ₹80,488,269.26


In [10]:
with gr.Blocks(title="Box Office Revenue Predictor") as app:

    gr.Markdown(
        """
        # 🎬 Box Office Revenue Predictor

        Predict the expected box-office revenue of an Indian film
        using Elastic Net Regression.
        """
    )

    with gr.Row():

        with gr.Column():

            release_period_input = gr.Dropdown(
                choices=release_periods,
                label="Release Period",
                value=release_periods[0]
            )

            remake_input = gr.Dropdown(
                choices=["Yes", "No"],
                label="Whether Remake",
                value="No"
            )

            franchise_input = gr.Dropdown(
                choices=["Yes", "No"],
                label="Whether Franchise",
                value="No"
            )

            genre_input = gr.Dropdown(
                choices=genres,
                label="Genre",
                value=genres[0]
            )

            new_actor_input = gr.Dropdown(
                choices=["Yes", "No"],
                label="New Actor",
                value="No"
            )

            new_director_input = gr.Dropdown(
                choices=["Yes", "No"],
                label="New Director",
                value="No"
            )

        with gr.Column():

            new_music_director_input = gr.Dropdown(
                choices=["Yes", "No"],
                label="New Music Director",
                value="No"
            )

            lead_star_input = gr.Dropdown(
                choices=lead_stars,
                label="Lead Star",
                value=lead_stars[0]
            )

            director_input = gr.Dropdown(
                choices=directors,
                label="Director",
                value=directors[0]
            )

            music_director_input = gr.Dropdown(
                choices=music_directors,
                label="Music Director",
                value=music_directors[0]
            )

            screens_input = gr.Number(
                label="Number of Screens",
                value=5
            )

            budget_input = gr.Number(
                label="Budget (INR)",
                value=85000
            )

    predict_button = gr.Button(
        "Predict Box Office Revenue"
    )

    output = gr.Textbox(
        label="Predicted Revenue"
    )

    predict_button.click(
        fn=predict_revenue,
        inputs=[
            release_period_input,
            remake_input,
            franchise_input,
            genre_input,
            new_actor_input,
            new_director_input,
            new_music_director_input,
            lead_star_input,
            director_input,
            music_director_input,
            screens_input,
            budget_input
        ],
        outputs=output
    )

In [11]:
app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [12]:
print("==========================================")
print("       DEPLOYMENT COMPLETED")
print("==========================================")

print("Model: Elastic Net Regression")
print("Target: Revenue(INR)")
print("Interface: Gradio")
print("Model path:", model_path)
print("Number of input features:", len(feature_columns))

       DEPLOYMENT COMPLETED
Model: Elastic Net Regression
Target: Revenue(INR)
Interface: Gradio
Model path: ../models/final_elastic_net_model.pkl
Number of input features: 2460
